# 메소드 비교 평가: 모든 언어에서 두 메소드 모두 동일 언어로 응답한 경우

이 노트북은 두 개의 메소드를 **6개 언어(zh, es, ko, th, sw, te)** 모두에 대해 비교합니다.

**핵심 로직**: 각 언어 질문에서 두 메소드 **모두** 해당 언어로 응답한 경우에 대해서만 평가를 진행합니다.

**주의**: 두 파일의 언어 순서가 다를 수 있으므로, 언어별로 데이터를 분리한 후 매칭합니다.

**⚠️ 커널 확인**: 이 노트북은 `.fasttext` 커널에서 실행되어야 합니다!

In [ ]:
# ========================================
# 설정 (Configuration)
#========================================

# 비교할 두 개의 메소드 suffix
METHOD_A = "-cot-Google-transQ"
METHOD_B = "-skeleton_multiturn-Google-transQ"


# 평가할 모델 리스트
MODEL_LIST = [
    "Qwen2.5-7B-Instruct",
    "Qwen2.5-14B-Instruct",
    # "Qwen2.5-32B-Instruct",
    "Qwen2.5-72B-Instruct",
    # "Meta-Llama-3.1-8B-Instruct",
    # "Meta-Llama-3.1-70B-Instruct",
]

# 벤치마크 설정
BENCHMARK_CONFIG = {
    "math-500": {"dir": "MATH-500-translated", "chunk_size": 500},
    "polymath": {"dir": "PolyMath-translated", "chunk_size": 500},
    "aime25": {"dir": "AIME25-translated", "chunk_size": 30},
    "mgsm": {"dir": "SSS", "chunk_size": 250},
    # "mgsm": {"dir": "MGSM", "chunk_size": 250},
}

# 평가할 벤치마크 선택
TARGET_BENCHMARK = "polymath"  # "math-500", "polymath", "aime25"
# TARGET_BENCHMARK = "math-500"  # "math-500", "polymath", "aime25"
# TARGET_BENCHMARK = "mgsm" 

# 비교할 6개 언어 리스트
# TARGET_LANGUAGES = ["fr"]
TARGET_LANGUAGES = ["zh", "es", "ko", "th", "sw"]
# TARGET_LANGUAGES = ["zh", "es", "ko", "th", "sw", "te"]

# 언어명 매핑
LANGUAGE_NAMES = {
    "zh": "Chinese",
    "es": "Spanish",
    "ko": "Korean",
    "th": "Thai",
    "sw": "Swahili",
    # "te": "Telugu",
}
# LANGUAGE_NAMES = {
#     "fr": "French",
# }

# 프로젝트 경로
PRJ_PATH = "/home/seoultech/MLP/hslim/SCALE/data/inference_result/eval"

# FastText 모델 경로
FASTTEXT_MODEL_PATH = "/home/seoultech/MLP/hslim/SCALE/lid.176.bin"

print(f"Method A: {METHOD_A}")
print(f"Method B: {METHOD_B}")
print(f"Models: {len(MODEL_LIST)} models")
print(f"Benchmark: {TARGET_BENCHMARK}")
print(f"Target Languages: {TARGET_LANGUAGES}")

Method A: -cot-Google-transQ
Method B: -skeleton_multiturn3-Google-transQ
Models: 3 models
Benchmark: polymath
Target Languages: ['zh', 'es', 'ko', 'th', 'sw']


In [12]:
#========================================
# Imports & FastText 모델 로드
#========================================

import fasttext
import json
import os
from collections import defaultdict, Counter
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
from math_verify import parse, verify
import traceback

# FastText 모델 로드
fasttext.FastText.eprint = lambda x: None  # Suppress warnings
lang_model = fasttext.load_model(FASTTEXT_MODEL_PATH)
print("FastText model loaded successfully")

# 테스트: 언어 감지가 제대로 되는지 확인
test_texts = {
    "zh": "这是一个测试文本",
    "ko": "이것은 테스트 텍스트입니다",
    "es": "Este es un texto de prueba",
}

print("\n🔍 FastText 언어 감지 테스트:")
for expected_lang, text in test_texts.items():
    labels, probs = lang_model.predict(text, k=1)
    detected = labels[0].replace("__label__", "")
    status = "✅" if detected == expected_lang else "❌"
    print(f"   {status} Expected: {expected_lang}, Detected: {detected}")

FastText model loaded successfully

🔍 FastText 언어 감지 테스트:
   ✅ Expected: zh, Detected: zh
   ✅ Expected: ko, Detected: ko
   ✅ Expected: es, Detected: es


In [13]:
#========================================
# Helper Functions
#========================================

def make_path(prj_path, benchmark_dir, model_path, suffix):
    """경로 생성 함수"""
    return f"{prj_path}/{benchmark_dir}/{model_path}{suffix}.jsonl"


def detect_language(text):
    """FastText를 사용하여 언어 감지"""
    global lang_model
    
    if text is None or not isinstance(text, str):
        return "unk"
    
    text = text.replace("\n", " ").strip()
    
    if len(text) < 2:
        return "unk"
    
    try:
        
        labels, probs = lang_model.predict(text, k=1)
        if not probs or len(probs) == 0:
            return "unk"
        return labels[0].replace("__label__", "")
    except Exception as e:
        print(f"⚠️ Language detection error: {e}")
        return "unk"


def load_jsonl(filepath):
    """JSONL 파일 로드"""
    data = []
    if not os.path.exists(filepath):
        print(f"⚠️ File not found: {filepath}")
        return None
    
    with open(filepath, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"JSON error on line {i}: {e}")
    return data


def add_detected_language(data, show_progress=False):
    """데이터에 detected_response_language 필드 추가"""
    iterator = tqdm(data, desc="   Detecting languages") if show_progress else data
    
    for item in iterator:
        # responses = item.get('response', [])
        # item['detected_response_language'] = detect_language(responses)
        responses = item.get('responses', [])
        if responses and isinstance(responses[0], str):
            item['detected_response_language'] = detect_language(responses[0])
        else:
            item['detected_response_language'] = "unk"
    return data


def group_by_language(data):
    """데이터를 언어별로 그룹화하여 딕셔너리로 반환"""
    grouped = defaultdict(list)
    for item in data:
        lang = item.get('question_language', 'unknown')
        grouped[lang].append(item)
    return grouped


print("Helper functions defined.")

Helper functions defined.


In [14]:
#========================================
# 메소드 비교 함수
#========================================

def evaluate_single_example(args):
    """단일 예제 평가 (multiprocessing용)"""
    example_a, example_b = args
    
    # 응답 가져오기
    response_a = example_a['responses'][0] if example_a.get('responses') else ""
    response_b = example_b['responses'][0] if example_b.get('responses') else ""
    
    # response_a = example_a['response'] if example_a.get('response') else ""
    # response_b = example_b['response'] if example_b.get('response') else ""
    try:
        gold = parse(str(example_a['answer']))
        
        # Method A 평가
        pred_a = parse(str(response_a))
        correct_a = int(verify(gold, pred_a))
        
        # Method B 평가
        pred_b = parse(str(response_b))
        correct_b = int(verify(gold, pred_b))
        
        return correct_a, correct_b, 0
    
    except Exception as e:
        return 0, 0, 1


def compare_methods_on_language(
    data_a_lang,  # 특정 언어의 data_a 리스트
    data_b_lang,  # 특정 언어의 data_b 리스트
    target_language,
    num_workers=None,
    verbose=True
):
    """
    특정 언어에 대해 두 메소드를 비교합니다.
    
    조건: 두 메소드 모두 해당 언어로 응답한 경우만 평가
    """
    if num_workers is None:
        num_workers = cpu_count()
    
    if not data_a_lang or not data_b_lang:
        if verbose:
            print(f"      [{target_language}] No data available")
        return None
    
    # 데이터 길이 맞추기 (짧은 쪽에 맞춤)
    min_len = min(len(data_a_lang), len(data_b_lang))
    data_a_lang = data_a_lang[:min_len]
    data_b_lang = data_b_lang[:min_len]
    
    # 필터링: 두 메소드 모두 타겟 언어로 응답한 경우만
    filtered_pairs = []
    lang_mismatch_a = 0
    lang_mismatch_b = 0
    
    for item_a, item_b in zip(data_a_lang, data_b_lang):
        resp_lang_a = item_a.get('detected_response_language', 'unk')
        resp_lang_b = item_b.get('detected_response_language', 'unk')
        
        if resp_lang_a != target_language:
            lang_mismatch_a += 1
            continue
        
        if resp_lang_b != target_language:
            lang_mismatch_b += 1
            continue
        
        filtered_pairs.append((item_a, item_b))
    
    total_target_questions = min_len
    
    if verbose:
        print(f"      [{target_language}] Total: {total_target_questions}, "
              f"A mismatch: {lang_mismatch_a}, B mismatch: {lang_mismatch_b}, "
              f"Both match: {len(filtered_pairs)}")
    
    if not filtered_pairs:
        return None
    
    # 병렬 평가
    with Pool(num_workers) as pool:
        if verbose:
            results = list(tqdm(
                pool.imap(evaluate_single_example, filtered_pairs),
                total=len(filtered_pairs),
                desc=f"      Evaluating {target_language}",
                leave=False
            ))
        else:
            results = pool.map(evaluate_single_example, filtered_pairs)
    
    # 결과 집계
    total = len(results)
    correct_a = sum(r[0] for r in results)
    correct_b = sum(r[1] for r in results)
    errors = sum(r[2] for r in results)
    
    acc_a = correct_a / total * 100 if total > 0 else 0
    acc_b = correct_b / total * 100 if total > 0 else 0
    
    return {
        "total_evaluated": total,
        "correct_a": correct_a,
        "correct_b": correct_b,
        "acc_a": acc_a,
        "acc_b": acc_b,
        "errors": errors,
        "total_target_questions": total_target_questions,
        "lang_mismatch_a": lang_mismatch_a,
        "lang_mismatch_b": lang_mismatch_b,
    }


print("Comparison function defined.")

Comparison function defined.


In [15]:
#========================================
# 메인 실행: 모든 모델 × 모든 언어 비교
# (언어별로 데이터를 그룹화하여 매칭)
#========================================

benchmark_dir = BENCHMARK_CONFIG[TARGET_BENCHMARK]["dir"]

# 결과 저장: all_results[model][language] = result
all_results = defaultdict(dict)

for model in MODEL_LIST:
    print(f"\n{'='*70}")
    print(f"🔍 Model: {model}")
    print(f"{'='*70}")
    
    # 파일 경로 생성
    path_a = make_path(PRJ_PATH, benchmark_dir, model, METHOD_A)
    path_b = make_path(PRJ_PATH, benchmark_dir, model, METHOD_B)
    
    print(f"   Method A: {os.path.basename(path_a)}")
    print(f"   Method B: {os.path.basename(path_b)}")
    
    # 데이터 로드
    data_a = load_jsonl(path_a)
    data_b = load_jsonl(path_b)
    
    if data_a is None or data_b is None:
        print(f"   ⚠️ Skipping {model} - file not found")
        continue
    
    print(f"   Loaded: A={len(data_a)}, B={len(data_b)} examples")
    
    # 언어 감지 추가 (진행 표시)
    print(f"   Detecting response languages for A...")
    data_a = add_detected_language(data_a, show_progress=True)
    print(f"   Detecting response languages for B...")
    data_b = add_detected_language(data_b, show_progress=True)
    
    # 언어별로 데이터 그룹화
    grouped_a = group_by_language(data_a)
    grouped_b = group_by_language(data_b)
    
    print(f"   Languages in A: {list(grouped_a.keys())}")
    print(f"   Languages in B: {list(grouped_b.keys())}")
    
    # 디버그: 첫 번째 언어의 감지 결과 확인
    first_lang = TARGET_LANGUAGES[0]
    if first_lang in grouped_a:
        sample_a = grouped_a[first_lang][:10]
        detected_a = Counter([item.get('detected_response_language') for item in sample_a])
        print(f"   DEBUG [{first_lang}] A detected langs (first 10): {dict(detected_a)}")
    if first_lang in grouped_b:
        sample_b = grouped_b[first_lang][:10]
        detected_b = Counter([item.get('detected_response_language') for item in sample_b])
        print(f"   DEBUG [{first_lang}] B detected langs (first 10): {dict(detected_b)}")
    
    # 각 언어에 대해 비교 수행
    print(f"\n   Comparing across target languages:")
    for lang in TARGET_LANGUAGES:
        data_a_lang = grouped_a.get(lang, [])
        data_b_lang = grouped_b.get(lang, [])
        
        result = compare_methods_on_language(
            data_a_lang, 
            data_b_lang, 
            target_language=lang,
            verbose=True
        )
        
        if result:
            all_results[model][lang] = result

print(f"\n\n{'='*70}")
print("✅ All comparisons completed!")


🔍 Model: Qwen2.5-7B-Instruct
   Method A: Qwen2.5-7B-Instruct-cot-Google-transQ.jsonl
   Method B: Qwen2.5-7B-Instruct-skeleton_multiturn3-Google-transQ.jsonl
   Loaded: A=3000, B=3000 examples
   Detecting response languages for A...


   Detecting languages: 100%|██████████| 3000/3000 [00:00<00:00, 3718.83it/s] 


   Detecting response languages for B...


   Detecting languages: 100%|██████████| 3000/3000 [00:00<00:00, 3666.76it/s] 

   Languages in A: ['zh', 'es', 'ko', 'th', 'sw', 'te']
   Languages in B: ['zh', 'es', 'ko', 'th', 'sw', 'te']
   DEBUG [zh] A detected langs (first 10): {'zh': 9, 'en': 1}
   DEBUG [zh] B detected langs (first 10): {'zh': 10}

   Comparing across target languages:
      [zh] Total: 500, A mismatch: 10, B mismatch: 9, Both match: 481


...

      [ko] Total: 500, A mismatch: 3, B mismatch: 9, Both match: 488


      [th] Total: 500, A mismatch: 3, B mismatch: 8, Both match: 489


      [sw] Total: 500, A mismatch: 93, B mismatch: 51, Both match: 356


      Evaluating sw:   1%|          | 3/356 [00:00<01:16,  4.59it/s]Timeout during comparison




✅ All comparisons completed!


In [16]:
#========================================
# 결과 요약: 언어별 테이블
#========================================

print(f"\n{'='*100}")
print(f"📊 결과 요약: 각 언어에서 두 메소드 모두 해당 언어로 응답한 경우")
print(f"   Benchmark: {TARGET_BENCHMARK}")
print(f"   Method A: {METHOD_A}")
print(f"   Method B: {METHOD_B}")
print(f"{'='*100}\n")

# 각 언어별로 테이블 출력
for lang in TARGET_LANGUAGES:
    lang_name = LANGUAGE_NAMES.get(lang, lang)
    print(f"\n{'─'*80}")
    print(f"🌐 {lang.upper()} ({lang_name})")
    print(f"{'─'*80}")
    
    print(f"{'Model':<35} {'N':>6} {'A_Acc':>10} {'B_Acc':>10} {'Diff':>10}")
    print(f"{'-'*35} {'-'*6} {'-'*10} {'-'*10} {'-'*10}")
    
    acc_a_list = []
    acc_b_list = []
    
    for model in MODEL_LIST:
        if model in all_results and lang in all_results[model]:
            result = all_results[model][lang]
            diff = result['acc_b'] - result['acc_a']
            diff_str = f"{diff:+.2f}%"
            
            print(f"{model:<35} {result['total_evaluated']:>6} "
                  f"{result['acc_a']:>9.2f}% {result['acc_b']:>9.2f}% {diff_str:>10}")
            
            acc_a_list.append(result['acc_a'])
            acc_b_list.append(result['acc_b'])
        else:
            print(f"{model:<35} {'N/A':>6} {'N/A':>10} {'N/A':>10} {'N/A':>10}")
    
    # 평균 계산
    if acc_a_list and acc_b_list:
        avg_a = sum(acc_a_list) / len(acc_a_list)
        avg_b = sum(acc_b_list) / len(acc_b_list)
        avg_diff = avg_b - avg_a
        
        print(f"{'-'*35} {'-'*6} {'-'*10} {'-'*10} {'-'*10}")
        print(f"{'AVERAGE':<35} {'':>6} {avg_a:>9.2f}% {avg_b:>9.2f}% {avg_diff:+.2f}%")


📊 결과 요약: 각 언어에서 두 메소드 모두 해당 언어로 응답한 경우
   Benchmark: polymath
   Method A: -cot-Google-transQ
   Method B: -skeleton_multiturn3-Google-transQ


────────────────────────────────────────────────────────────────────────────────
🌐 ZH (Chinese)
────────────────────────────────────────────────────────────────────────────────
Model                                    N      A_Acc      B_Acc       Diff
----------------------------------- ------ ---------- ---------- ----------
Qwen2.5-7B-Instruct                    481     29.52%     32.02%     +2.49%
Qwen2.5-14B-Instruct                   491     31.98%     33.20%     +1.22%
Qwen2.5-72B-Instruct                   481     34.51%     34.30%     -0.21%
----------------------------------- ------ ---------- ---------- ----------
AVERAGE                                        32.00%     33.17% +1.17%

────────────────────────────────────────────────────────────────────────────────
🌐 ES (Spanish)
─────────────────────────────────────────────────────

In [17]:
#========================================
# 결과 요약: 모델별 전체 언어 비교 테이블
#========================================

print(f"\n{'='*120}")
print(f"📊 모델별 전체 언어 비교 (Method B - Method A 차이)")
print(f"{'='*120}\n")

# 헤더
header = f"{'Model':<35}"
for lang in TARGET_LANGUAGES:
    header += f" {lang.upper():>10}"
header += f" {'AVG':>10}"
print(header)
print(f"{'-'*35}" + f" {'-'*10}" * (len(TARGET_LANGUAGES) + 1))

# 각 모델별 결과
for model in MODEL_LIST:
    row = f"{model:<35}"
    diffs = []
    
    for lang in TARGET_LANGUAGES:
        if model in all_results and lang in all_results[model]:
            result = all_results[model][lang]
            diff = result['acc_b'] - result['acc_a']
            diffs.append(diff)
            row += f" {diff:>+9.2f}%"
        else:
            row += f" {'N/A':>10}"
    
    # 평균 차이
    if diffs:
        avg_diff = sum(diffs) / len(diffs)
        row += f" {avg_diff:>+9.2f}%"
    else:
        row += f" {'N/A':>10}"
    
    print(row)

# 전체 평균
print(f"{'-'*35}" + f" {'-'*10}" * (len(TARGET_LANGUAGES) + 1))
avg_row = f"{'OVERALL AVG':<35}"
overall_diffs = []

for lang in TARGET_LANGUAGES:
    lang_diffs = []
    for model in MODEL_LIST:
        if model in all_results and lang in all_results[model]:
            result = all_results[model][lang]
            lang_diffs.append(result['acc_b'] - result['acc_a'])
    
    if lang_diffs:
        lang_avg = sum(lang_diffs) / len(lang_diffs)
        avg_row += f" {lang_avg:>+9.2f}%"
        overall_diffs.append(lang_avg)
    else:
        avg_row += f" {'N/A':>10}"

if overall_diffs:
    overall_avg = sum(overall_diffs) / len(overall_diffs)
    avg_row += f" {overall_avg:>+9.2f}%"
else:
    avg_row += f" {'N/A':>10}"

print(avg_row)


📊 모델별 전체 언어 비교 (Method B - Method A 차이)

Model                                       ZH         ES         KO         TH         SW        AVG
----------------------------------- ---------- ---------- ---------- ---------- ---------- ----------
Qwen2.5-7B-Instruct                     +2.49%     +2.30%     +0.22%     +1.07%     -0.58%     +1.10%
Qwen2.5-14B-Instruct                    +1.22%     -1.02%     +2.06%     +0.00%     +2.51%     +0.95%
Qwen2.5-72B-Instruct                    -0.21%     +1.83%     +0.82%     -2.45%     -1.97%     -0.40%
----------------------------------- ---------- ---------- ---------- ---------- ---------- ----------
OVERALL AVG                             +1.17%     +1.03%     +1.03%     -0.46%     -0.01%     +0.55%


In [18]:
#========================================
# Method A Accuracy 테이블
#========================================

print(f"\n{'='*120}")
print(f"📊 Method A Accuracy ({METHOD_A})")
print(f"{'='*120}\n")

header = f"{'Model':<35}"
for lang in TARGET_LANGUAGES:
    header += f" {lang.upper():>10}"
header += f" {'AVG':>10}"
print(header)
print(f"{'-'*35}" + f" {'-'*10}" * (len(TARGET_LANGUAGES) + 1))

for model in MODEL_LIST:
    row = f"{model:<35}"
    accs = []
    
    for lang in TARGET_LANGUAGES:
        if model in all_results and lang in all_results[model]:
            acc = all_results[model][lang]['acc_a']
            accs.append(acc)
            row += f" {acc:>9.2f}%"
        else:
            row += f" {'N/A':>10}"
    
    if accs:
        avg = sum(accs) / len(accs)
        row += f" {avg:>9.2f}%"
    else:
        row += f" {'N/A':>10}"
    
    print(row)

print(f"\n{'='*120}")
print(f"📊 Method B Accuracy ({METHOD_B})")
print(f"{'='*120}\n")

header = f"{'Model':<35}"
for lang in TARGET_LANGUAGES:
    header += f" {lang.upper():>10}"
header += f" {'AVG':>10}"
print(header)
print(f"{'-'*35}" + f" {'-'*10}" * (len(TARGET_LANGUAGES) + 1))

for model in MODEL_LIST:
    row = f"{model:<35}"
    accs = []
    
    for lang in TARGET_LANGUAGES:
        if model in all_results and lang in all_results[model]:
            acc = all_results[model][lang]['acc_b']
            accs.append(acc)
            row += f" {acc:>9.2f}%"
        else:
            row += f" {'N/A':>10}"
    
    if accs:
        avg = sum(accs) / len(accs)
        row += f" {avg:>9.2f}%"
    else:
        row += f" {'N/A':>10}"
    
    print(row)


📊 Method A Accuracy (-cot-Google-transQ)

Model                                       ZH         ES         KO         TH         SW        AVG
----------------------------------- ---------- ---------- ---------- ---------- ---------- ----------
Qwen2.5-7B-Instruct                     29.52%     31.73%     27.65%     29.00%     16.76%     26.93%
Qwen2.5-14B-Instruct                    31.98%     35.86%     29.90%     31.51%     21.80%     30.21%
Qwen2.5-72B-Instruct                    34.51%     35.64%     34.63%     35.38%     35.67%     35.17%

📊 Method B Accuracy (-skeleton_multiturn3-Google-transQ)

Model                                       ZH         ES         KO         TH         SW        AVG
----------------------------------- ---------- ---------- ---------- ---------- ---------- ----------
Qwen2.5-7B-Instruct                     32.02%     34.03%     27.86%     30.06%     16.18%     28.03%
Qwen2.5-14B-Instruct                    33.20%     34.84%     31.96%     31.51%   

In [19]:
#========================================
# 언어 불일치 상세 분석
#========================================

print(f"\n{'='*100}")
print(f"📊 언어 불일치 분석: 평가에서 제외된 샘플 수")
print(f"{'='*100}\n")

for lang in TARGET_LANGUAGES:
    lang_name = LANGUAGE_NAMES.get(lang, lang)
    print(f"\n{'─'*80}")
    print(f"🌐 {lang.upper()} ({lang_name})")
    print(f"{'─'*80}")
    
    print(f"{'Model':<35} {'Total':>8} {'A !match':>10} {'B !match':>10} {'Evaluated':>10}")
    print(f"{'-'*35} {'-'*8} {'-'*10} {'-'*10} {'-'*10}")
    
    for model in MODEL_LIST:
        if model in all_results and lang in all_results[model]:
            result = all_results[model][lang]
            print(f"{model:<35} "
                  f"{result['total_target_questions']:>8} "
                  f"{result['lang_mismatch_a']:>10} "
                  f"{result['lang_mismatch_b']:>10} "
                  f"{result['total_evaluated']:>10}")
        else:
            print(f"{model:<35} {'N/A':>8} {'N/A':>10} {'N/A':>10} {'N/A':>10}")

print(f"\n* 'A !match': Method A가 해당 언어가 아닌 다른 언어로 응답한 횟수")
print(f"* 'B !match': Method B가 해당 언어가 아닌 다른 언어로 응답한 횟수")
print(f"* 'Evaluated': 두 메소드 모두 해당 언어로 응답하여 평가된 횟수")


📊 언어 불일치 분석: 평가에서 제외된 샘플 수


────────────────────────────────────────────────────────────────────────────────
🌐 ZH (Chinese)
────────────────────────────────────────────────────────────────────────────────
Model                                  Total   A !match   B !match  Evaluated
----------------------------------- -------- ---------- ---------- ----------
Qwen2.5-7B-Instruct                      500         10          9        481
Qwen2.5-14B-Instruct                     500          7          2        491
Qwen2.5-72B-Instruct                     500          8         11        481

────────────────────────────────────────────────────────────────────────────────
🌐 ES (Spanish)
────────────────────────────────────────────────────────────────────────────────
Model                                  Total   A !match   B !match  Evaluated
----------------------------------- -------- ---------- ---------- ----------
Qwen2.5-7B-Instruct                      500         13          8   

In [20]:
#========================================
# 난이도별 분석 (polymath)
#========================================

def analyze_by_difficulty(data_a_lang, data_b_lang, target_language, num_workers=None):
    """
    난이도별로 두 메소드를 비교합니다.
    """
    if num_workers is None:
        num_workers = cpu_count()
    
    if not data_a_lang or not data_b_lang:
        return {}
    
    # 데이터 길이 맞추기
    min_len = min(len(data_a_lang), len(data_b_lang))
    data_a_lang_sliced = data_a_lang[:min_len]
    data_b_lang_sliced = data_b_lang[:min_len]
    
    # 난이도별 그룹화
    difficulty_groups = defaultdict(list)
    
    for i, (item_a, item_b) in enumerate(zip(data_a_lang_sliced, data_b_lang_sliced)):
        # 언어 체크
        resp_lang_a = item_a.get('detected_response_language', 'unk')
        resp_lang_b = item_b.get('detected_response_language', 'unk')

        if resp_lang_a != target_language or resp_lang_b != target_language:
            continue
        
        # 난이도 확인
        difficulty = item_a.get('difficulty', 'unknown')
        difficulty_groups[difficulty].append((item_a, item_b))
    
    results = {}
    
    for diff, pairs in difficulty_groups.items():
        if not pairs:
            continue
            
        # 병렬 평가
        with Pool(num_workers) as pool:
            eval_results = pool.map(evaluate_single_example, pairs)
        
        total = len(eval_results)
        correct_a = sum(r[0] for r in eval_results)
        correct_b = sum(r[1] for r in eval_results)
        
        acc_a = correct_a / total * 100 if total > 0 else 0
        acc_b = correct_b / total * 100 if total > 0 else 0
        
        results[diff] = {
            'total': total,
            'correct_a': correct_a,
            'correct_b': correct_b,
            'acc_a': acc_a,
            'acc_b': acc_b
        }
    return results

print("\n" + "="*70)
print("📊 Difficulty Analysis (PolyMath)")
print("="*70)

# 난이도 순서 (출력용)
DIFF_ORDER = ['easy', 'medium', 'hard', 'top']

for model in MODEL_LIST:
    print(f"\n🔍 Model: {model}")
    
    # 파일 경로 생성
    path_a = make_path(PRJ_PATH, benchmark_dir, model, METHOD_A)
    path_b = make_path(PRJ_PATH, benchmark_dir, model, METHOD_B)
    
    # 데이터 로드
    data_a = load_jsonl(path_a)
    data_b = load_jsonl(path_b)
    
    if not data_a or not data_b:
        print(f"   ⚠️ Data not found for {model}")
        continue
        
    print("   Detecting languages...")
    data_a = add_detected_language(data_a)
    data_b = add_detected_language(data_b)
    
    grouped_a = group_by_language(data_a)
    grouped_b = group_by_language(data_b)
    
    print(f"   {'Language':<10} {'Difficulty':<10} {'Count':<8} {'Method A':<10} {'Method B':<10} {'Diff (B-A)':<10}")
    print(f"   {'-'*75}")
    
    for lang in TARGET_LANGUAGES:
        lang_results = analyze_by_difficulty(
            grouped_a[lang],
            grouped_b[lang],
            target_language=lang
        )
        
        if not lang_results:
            print(f"   {lang:<10} {'(no data)':<10}")
            continue
            
        # 정해진 순서대로 출력, 없으면 건너뜀
        available_diffs = set(lang_results.keys())
        sorted_diffs = [d for d in DIFF_ORDER if d in available_diffs] + [d for d in available_diffs if d not in DIFF_ORDER]
        
        for i, diff in enumerate(sorted_diffs):
            res = lang_results[diff]
            acc_a = res['acc_a']
            acc_b = res['acc_b']
            diff_val = acc_b - acc_a
            
            lang_str = lang if i == 0 else ""
            print(f"   {lang_str:<10} {diff:<10} {res['total']:<8} {acc_a:6.2f}%    {acc_b:6.2f}%    {diff_val:+6.2f}%")
        print(f"   {'-'*75}")



📊 Difficulty Analysis (PolyMath)

🔍 Model: Qwen2.5-7B-Instruct
   Detecting languages...
   Language   Difficulty Count    Method A   Method B   Diff (B-A)
   ---------------------------------------------------------------------------


Timeout during comparison


   zh         medium     117       23.93%     29.06%     +5.13%
              top        120        3.33%      5.00%     +1.67%
              low        125       80.00%     82.40%     +2.40%
              high       119        8.40%      9.24%     +0.84%
   ---------------------------------------------------------------------------
   es         medium     119       26.05%     26.89%     +0.84%
              top        119        5.88%     10.08%     +4.20%
              low        125       84.00%     84.80%     +0.80%
              high       116        7.76%     11.21%     +3.45%
   ---------------------------------------------------------------------------
   ko         medium     114       21.93%     22.81%     +0.88%
              top        113        4.42%      2.65%     -1.77%
              low        120       75.00%     75.83%     +0.83%
              high       116        6.90%      7.76%     +0.86%
   -----------------------------------------------------------------------

   sw         medium     63         1.59%      3.17%     +1.59%
              top        91         2.20%      0.00%     -2.20%
              low        102       52.94%     51.96%     -0.98%
              high       90         1.11%      1.11%     +0.00%
   ---------------------------------------------------------------------------

🔍 Model: Qwen2.5-14B-Instruct
   Detecting languages...
   Language   Difficulty Count    Method A   Method B   Diff (B-A)
   ---------------------------------------------------------------------------
   zh         medium     122       27.87%     27.87%     +0.00%
              top        122        9.84%      7.38%     -2.46%
              low        125       80.80%     85.60%     +4.80%
              high       122        8.20%     10.66%     +2.46%
   ---------------------------------------------------------------------------
   es         medium     120       34.17%     29.17%     -5.00%
              top        121        8.26%      7.44%     -0.83%

Timeout during comparison


   sw         medium     52        23.08%     19.23%     -3.85%
              top        95         5.26%      4.21%     -1.05%
              low        124       83.87%     83.06%     -0.81%
              high       85         7.06%      3.53%     -3.53%
   ---------------------------------------------------------------------------


In [21]:
#========================================
# McNemar's Test for MGSM (Low Resource)
#========================================

import numpy as np
from scipy import stats

def perform_mcnemar_test(data_a, data_b, target_langs=['th', 'sw', 'te', 'bn', 'my', 'yo']):
    """
    특정 언어들에 대해 McNemar 검정을 수행합니다.
    """
    if not data_a or not data_b:
        return None
    
    # 언어별로 필터링하여 데이터 모으기
    filtered_pairs = []
    
    min_len = min(len(data_a), len(data_b))
    data_a = data_a[:min_len]
    data_b = data_b[:min_len]
    
    for item_a, item_b in zip(data_a, data_b):
        lang = item_a.get('question_language')
        if lang in target_langs:
            filtered_pairs.append((item_a, item_b))
            
    if not filtered_pairs:
        return {'b': 0, 'c': 0, 'total': 0, 'p_value': 1.0, 'statistic': 0.0}

    # 정확도 평가
    with Pool(cpu_count()) as pool:
        results = pool.map(evaluate_single_example, filtered_pairs)
        
    b = 0  # Method A correct, Method B incorrect
    c = 0  # Method A incorrect, Method B correct
    both_correct = 0
    both_incorrect = 0
    
    for res in results:
        correct_a, correct_b, _ = res
        
        if correct_a and not correct_b:
            b += 1
        elif not correct_a and correct_b:
            c += 1
        elif correct_a and correct_b:
            both_correct += 1
        else:
            both_incorrect += 1
            
    # McNemar's statistic calculation
    if b + c > 0:
        statistic = (abs(b - c) - 1)**2 / (b + c)
        p_value = stats.chi2.sf(statistic, 1)
    else:
        statistic = 0.0
        p_value = 1.0
        
    return {
        'b': b,
        'c': c,
        'both_correct': both_correct,
        'both_incorrect': both_incorrect,
        'total': len(filtered_pairs),
        'statistic': statistic,
        'p_value': p_value
    }

print("\n" + "="*70)
print("🧪 McNemar's Test Analysis (Low Resource MGSM)")
print("="*70)

# MGSM Low Resource Languages
LOW_RESOURCE_LANGS = ['th', 'sw', 'te', 'bn', 'my', 'yo']
print(f"Target Languages: {LOW_RESOURCE_LANGS}")

# MGSM 경로 설정
mgsm_benchmark_dir = BENCHMARK_CONFIG['mgsm']['dir']

for model in MODEL_LIST:
    print(f"\n🔍 Model: {model}")
    
    path_a = make_path(PRJ_PATH, mgsm_benchmark_dir, model, METHOD_A)
    path_b = make_path(PRJ_PATH, mgsm_benchmark_dir, model, METHOD_B)
    
    data_a = load_jsonl(path_a)
    data_b = load_jsonl(path_b)
    
    if not data_a or not data_b:
        print(f"   ⚠️ Data not found for {model} (MGSM)")
        continue
        
    result = perform_mcnemar_test(data_a, data_b, target_langs=LOW_RESOURCE_LANGS)
    
    if result and result['total'] > 0:
        print(f"   Total Samples (Low Resource): {result['total']}")
        print(f"   Contingency Table:")
        print(f"     Both Correct: {result['both_correct']}")
        print(f"     Both Incorrect: {result['both_incorrect']}")
        print(f"     Only Method A Correct (b): {result['b']}")
        print(f"     Only Method B Correct (c): {result['c']}")
        print(f"   McNemar's Statistic: {result['statistic']:.4f}")
        print(f"   P-value: {result['p_value']:.4e}")
        
        alpha = 0.05
        if result['p_value'] < alpha:
            better = "Method A" if result['b'] > result['c'] else "Method B"
            print(f"   ✅ Significant difference found (p < {alpha}). {better} performed better.")
        else:
            print(f"   ❌ No significant difference found (p >= {alpha}).")
    else:
        print(f"   ⚠️ No matching data found for target languages.")
    print(f"   {'-'*70}")



🧪 McNemar's Test Analysis (Low Resource MGSM)
Target Languages: ['th', 'sw', 'te', 'bn', 'my', 'yo']

🔍 Model: Qwen2.5-7B-Instruct
⚠️ File not found: /home/seoultech/MLP/hslim/SCALE/data/inference_result/eval/SSS/Qwen2.5-7B-Instruct-skeleton_multiturn3-Google-transQ.jsonl
   ⚠️ Data not found for Qwen2.5-7B-Instruct (MGSM)

🔍 Model: Qwen2.5-14B-Instruct
⚠️ File not found: /home/seoultech/MLP/hslim/SCALE/data/inference_result/eval/SSS/Qwen2.5-14B-Instruct-skeleton_multiturn3-Google-transQ.jsonl
   ⚠️ Data not found for Qwen2.5-14B-Instruct (MGSM)

🔍 Model: Qwen2.5-72B-Instruct
⚠️ File not found: /home/seoultech/MLP/hslim/SCALE/data/inference_result/eval/SSS/Qwen2.5-72B-Instruct-skeleton_multiturn3-Google-transQ.jsonl
   ⚠️ Data not found for Qwen2.5-72B-Instruct (MGSM)
